# FASE 3 - Procesamiento de Datos

## Paso 1: Importación de librerías y carga de datos

Importamos las librerías necesarias para el procesamiento matemático y matricial, y cargamos el dataset limpio proveniente de la Fase 2. También convertimos la fecha a formato `datetime` UTC.

In [10]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import warnings

# Ocultar warnings de Pandas
warnings.filterwarnings('ignore')

print("=> FASE DE PREPROCESAMIENTO...")

# Cargar dataset limpio de la fase anterior
ruta_limpio = '../../data/raw/2020-Apr-L.csv'
df_moda = pd.read_csv(ruta_limpio)

# Convertir a datetime UTC (necesario para el modelo temporal GRU)
df_moda['event_time'] = pd.to_datetime(df_moda['event_time'], utc=True)

# Verificamos que cargó bien (Para ver exactamente los 10 primeros registros)
df_moda.head(10)

=> FASE DE PREPROCESAMIENTO...


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-04-01 00:00:00+00:00,view,1201465,2232732101407408685,apparel.shoes.slipons,samsung,230.38,568984877,e2456cef-2d4f-42b9-a53a-8893cb0c6851
1,2020-04-01 00:00:03+00:00,view,9500109,2232732104175649385,apparel.scarf,defender,25.05,530206135,e3c1fb4b-0a7e-457d-a0cf-5d1479e9aafc
2,2020-04-01 00:00:05+00:00,view,1201295,2232732101407408685,apparel.shoes.slipons,apple,489.05,635165586,48d05455-e287-4c44-84f9-76621e02b612
3,2020-04-01 00:00:09+00:00,view,9200640,2232732104343421549,apparel.scarf,defender,20.19,533896443,6a220235-f4d6-4987-a51e-8f315b3027fc
4,2020-04-01 00:00:11+00:00,view,17301057,2232732098446229999,apparel.shoes.sandals,unknown,31.18,543165860,7eaf3210-1d5e-4abc-8437-ae54960c3570
5,2020-04-01 00:00:11+00:00,view,6000094,2232732103546503769,apparel.shoes,starline,127.42,514487953,cea7a5e7-af98-45b0-af3d-47d4ed3bd342
6,2020-04-01 00:00:14+00:00,view,30500022,2232732114929845090,apparel.shoes,hyundai,525.37,512576112,1d573a8f-7250-4f16-b2ad-342ca4440391
7,2020-04-01 00:00:15+00:00,view,4100184,2232732098228126185,apparel.shoes,sony,800.54,635165471,c6ce1b9b-19d1-429b-a7f4-3f72395c95b4
8,2020-04-01 00:00:19+00:00,view,17302960,2232732098446229999,apparel.shoes.sandals,dior,145.15,518083481,38c08bc4-399c-4389-bf15-f81b09417ced
9,2020-04-01 00:00:29+00:00,view,17301124,2232732098446229999,apparel.shoes.sandals,dior,99.10,518083481,38c08bc4-399c-4389-bf15-f81b09417ced


## Paso 2: Feature Engineering (Cálculo de Scores de Interacción)

Transformamos los datos crudos en variables de afinidad matemáticas. Asignamos pesos según el tipo de evento (vista, carrito, compra), calculamos la frecuencia de interacciones y la recencia usando un decaimiento exponencial para obtener un "score final" de afinidad usuario-producto.

In [11]:
print("Calculando scores de interacción (Frecuencia + Recencia)...")

# Asignar pesos por nivel de interés (vista=1, carrito=2, compra=3)
pesos = {'view': 1, 'cart': 2, 'purchase': 3}
df_moda['peso'] = df_moda['event_type'].map(pesos)

# Agrupar historial por usuario y producto para contar interacciones
interacciones = df_moda.groupby(['user_id', 'product_id']).agg(
    frecuencia=('event_type', 'count'),
    peso_total=('peso', 'sum'),
    ultima_interaccion=('event_time', 'max')
).reset_index()

# Calcular recencia: Decaimiento exponencial (interacciones antiguas valen menos)
fecha_actual = df_moda['event_time'].max()
lambda_decay = 0.1
interacciones['delta_t'] = (fecha_actual - interacciones['ultima_interaccion']).dt.days
interacciones['recencia'] = np.exp(-lambda_decay * interacciones['delta_t'])

# Score Final: Ecuación de afinidad (Peso + Frecuencia + Recencia)
interacciones['score_final'] = (
    interacciones['peso_total'] + 
    interacciones['frecuencia'] + 
    interacciones['recencia']
)
print(f"Total de interacciones únicas calculadas: {len(interacciones):,}")

interacciones.head()

Calculando scores de interacción (Frecuencia + Recencia)...
Total de interacciones únicas calculadas: 5,682,786


,user_id,product_id,frecuencia,peso_total,ultima_interaccion,delta_t,recencia,score_final
0,29515875,1201254,1,1,2020-04-23 05:03:44+00:00,7,0.496585,2.496585
1,29515875,1201256,1,1,2020-04-23 04:54:40+00:00,7,0.496585,2.496585
2,29515875,1201421,1,1,2020-04-23 05:01:43+00:00,7,0.496585,2.496585
3,29515875,1201564,1,1,2020-04-23 04:59:01+00:00,7,0.496585,2.496585
4,29515875,100044764,1,1,2020-04-23 04:50:24+00:00,7,0.496585,2.496585


## Paso 3: Construcción de la Matriz Dispersa (User-Item)

Construimos una matriz dispersa (formato CSR) para optimizar la memoria y no guardar ceros. Esta matriz relaciona a los usuarios con los productos utilizando los scores de interacción, sirviendo de insumo para los algoritmos de filtrado colaborativo.

In [12]:
print("\nConstruyendo la Matriz Usuario-Producto (Formato Disperso CSR)...")

usuarios_unicos = interacciones['user_id'].unique()
productos_unicos = interacciones['product_id'].unique()

# Mapear IDs reales a índices numéricos secuenciales (0, 1, 2...)
user_to_index = {user_id: idx for idx, user_id in enumerate(usuarios_unicos)}
product_to_index = {product_id: idx for idx, product_id in enumerate(productos_unicos)}

filas = interacciones['user_id'].map(user_to_index).values
columnas = interacciones['product_id'].map(product_to_index).values
valores = interacciones['score_final'].values

# Generar matriz insumo para el algoritmo KNN
matriz_interaccion_sparse = csr_matrix(
    (valores, (filas, columnas)), 
    shape=(len(usuarios_unicos), len(productos_unicos))
)

print(f"Dimensión Matriz Dispersa: {matriz_interaccion_sparse.shape}")
print(f"Consumo de RAM aproximado: {matriz_interaccion_sparse.data.nbytes / (1024**2):.2f} MB")


Construyendo la Matriz Usuario-Producto (Formato Disperso CSR)...
Dimensión Matriz Dispersa: (1379296, 60525)
Consumo de RAM aproximado: 43.36 MB


## Paso 4: Vectores de Producto (Filtrado por Contenido)

Preparamos las características de los productos (categoría, marca y precio) para que puedan ser procesadas por la red neuronal. Usamos `One-Hot Encoding` para las variables categóricas de texto y `Min-Max Scaling` para normalizar el precio a un rango entre 0 y 1.

In [13]:
print("\n=== PREPARANDO VECTORES DE CONTENIDO ===")

# Obtener catálogo único de productos
df_productos = df_moda[['product_id', 'category_code', 'brand', 'price']].drop_duplicates(subset=['product_id']).copy()

# 1. One-Hot Encoding: Convertir texto a columnas binarias (0s y 1s)
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoded = encoder.fit_transform(df_productos[['category_code', 'brand']])

# 2. Min-Max Scaling: Normalizar precio entre 0 y 1 para evitar sesgos
scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(df_productos[['price']])

# 3. Concatenar todo en la matriz final de características
vectores_productos = np.hstack((cat_encoded, price_scaled))

print(f"Total de productos procesados: {len(df_productos):,}")
print(f"Dimensión de la Matriz de Vectores de Contenido: {vectores_productos.shape}")
print("\n¡Preprocesamiento finalizado con éxito! Todos los tensores están listos para la arquitectura híbrida.")


=== PREPARANDO VECTORES DE CONTENIDO ===
Total de productos procesados: 60,525
Dimensión de la Matriz de Vectores de Contenido: (60525, 2117)

¡Preprocesamiento finalizado con éxito! Todos los tensores están listos para la arquitectura híbrida.


## Paso 5: Resultados del Procesamiento en Cuadros

Visualizamos los primeros registros de los scores de interacción y de los vectores de contenido procesados para validar que los pasos anteriores se hayan ejecutado correctamente.

In [14]:
print("=== CUADRO 1: SCORES DE INTERACCIÓN (Primeros 10 registros) ===")
display(interacciones.head(10))

print("\n=== CUADRO 2: VECTORES DE CONTENIDO (Primeros 10 productos) ===")
df_vectores = pd.DataFrame(vectores_productos[:10])
df_vectores.insert(0, 'product_id', df_productos['product_id'].values[:10])
display(df_vectores)

=== CUADRO 1: SCORES DE INTERACCIÓN (Primeros 10 registros) ===


,user_id,product_id,frecuencia,peso_total,ultima_interaccion,delta_t,recencia,score_final
0,29515875,1201254,1,1,2020-04-23 05:03:44+00:00,7,0.496585,2.496585
1,29515875,1201256,1,1,2020-04-23 04:54:40+00:00,7,0.496585,2.496585
2,29515875,1201421,1,1,2020-04-23 05:01:43+00:00,7,0.496585,2.496585
3,29515875,1201564,1,1,2020-04-23 04:59:01+00:00,7,0.496585,2.496585
4,29515875,100044764,1,1,2020-04-23 04:50:24+00:00,7,0.496585,2.496585
5,29515875,100083252,1,1,2020-04-14 02:20:44+00:00,16,0.201897,2.201897
6,42896738,100195589,1,1,2020-04-29 16:05:52+00:00,1,0.904837,2.904837
7,49484535,18800006,1,1,2020-04-18 11:16:05+00:00,12,0.301194,2.301194
8,49484535,18800010,1,1,2020-04-18 11:15:31+00:00,12,0.301194,2.301194
9,62336140,7900506,1,1,2020-04-15 16:15:56+00:00,15,0.223130,2.223130



=== CUADRO 2: VECTORES DE CONTENIDO (Primeros 10 productos) ===


,product_id,0,1,2,3,4,5,6,7,8,...,2107,2108,2109,2110,2111,2112,2113,2114,2115,2116
0,1201465,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.089463
1,9500109,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009683
2,1201295,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.189969
3,9200640,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.007794
4,17301057,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.012064
5,6000094,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.049458
6,30500022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.204081
7,4100184,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.310998
8,17302960,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.056347
9,17301124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.038455


In [15]:
# Guardar al final de procesamiento.ipynb
import os
os.makedirs('../deep_learning/datasets', exist_ok=True)
interacciones.to_csv('../../data/processed/interacciones_procesadas.csv', index=False)

## Generar testing_set.csv

In [16]:
# ==============================================================================
# PREPARACIÓN PARA EVALUACIÓN (Fase 5)
# Generación de Testing Set y Baseline
# ==============================================================================
import pandas as pd
from datetime import timedelta

print("--- 0. CARGANDO DATASET LIMPIO ---")
# Esto soluciona el NameError cargando tu dataset directamente
df = pd.read_csv('../../data/raw/2020-Apr-L.csv')
df['event_time'] = pd.to_datetime(df['event_time']) # Aseguramos formato fecha

print("\n--- 1. DIVISIÓN TEMPORAL (TRAIN / TEST) ---")
# Ordenar cronológicamente para respetar la secuencia del tiempo
df = df.sort_values('event_time')

# Definir el umbral temporal (ej. la última semana del dataset para testing)
fecha_maxima = df['event_time'].max()
fecha_umbral = fecha_maxima - pd.Timedelta(days=7)

# Partición de los datos
df_train = df[df['event_time'] < fecha_umbral].copy()
df_test = df[df['event_time'] >= fecha_umbral].copy()

print(f"Registros de entrenamiento: {len(df_train):,}")
print(f"Registros de prueba (Testing): {len(df_test):,}")

print("\n--- 2. CREACIÓN DEL GROUND TRUTH (testing_set.csv) ---")
# Mapear pesos de relevancia: purchase=3, cart=2, view=1
mapeo_relevancia = {'purchase': 3, 'cart': 2, 'view': 1}
df_test['relevancia'] = df_test['event_type'].map(mapeo_relevancia)

# Agrupar por usuario en el set de prueba
testing_set = df_test.groupby('user_id').apply(
    lambda x: pd.Series({
        'items_interactuados': x['product_id'].unique().tolist(),
        'diccionario_relevancia': dict(x.groupby('product_id')['relevancia'].max())
    })
).reset_index()

# Exportar el set de prueba
testing_set.to_csv('../../data/processed/testing_set.csv', index=False)
print(f"Exportado: '../../data/processed/testing_set.csv' con {len(testing_set)} usuarios de prueba.")

print("\n--- 3. CREACIÓN DEL BASELINE (predicciones_baseline.csv) ---")
# Obtener los 50 productos con más interacciones del set de ENTRENAMIENTO
top_productos_baseline = df_train['product_id'].value_counts().head(50).index.tolist()

# Asignar esta misma lista a todos los usuarios del set de pruebas
baseline_df = pd.DataFrame({
    'user_id': testing_set['user_id'],
    'recomendaciones': [top_productos_baseline for _ in range(len(testing_set))]
})

# Exportar las predicciones base
baseline_df.to_csv('../../data/processed/predicciones_baseline.csv', index=False)
print("Exportado: '../../data/processed/predicciones_baseline.csv' (Recomendador de 'Más Vendidos').")

print("\n--- 4. EXPORTAR SET DE ENTRENAMIENTO ---")
# Guardar el set de entrenamiento para que ml.ipynb y dl.ipynb lo consuman
df_train.to_csv('../../data/processed/2020-Apr-L_train.csv', index=False)
print("Exportado: '../../data/processed/2020-Apr-L_train.csv'")

--- 0. CARGANDO DATASET LIMPIO ---

--- 1. DIVISIÓN TEMPORAL (TRAIN / TEST) ---
Registros de entrenamiento: 6,922,944
Registros de prueba (Testing): 2,678,984

--- 2. CREACIÓN DEL GROUND TRUTH (testing_set.csv) ---
Exportado: '../../data/processed/testing_set.csv' con 484155 usuarios de prueba.

--- 3. CREACIÓN DEL BASELINE (predicciones_baseline.csv) ---
Exportado: '../../data/processed/predicciones_baseline.csv' (Recomendador de 'Más Vendidos').

--- 4. EXPORTAR SET DE ENTRENAMIENTO ---
Exportado: '../../data/processed/2020-Apr-L_train.csv'
